# ⚽ Mission 12: Soccer Vision Lab — Build an AI Tactical Analyst

## 📖 Mission Story
Welcome back! 🎉

During the first eleven missions, you gradually learned how to become an AI-powered soccer coach. You started with Python programming, then learned how to organize data with Pandas, visualize player statistics, build professional Streamlit dashboards, and even consult Gemini AI to answer tactical questions.

But there has always been one important limitation: every dashboard you built assumed that someone had already collected the player statistics (e.g., Artin_FC_v2 dataset). Professional soccer clubs don't work that way. Instead of receiving ready-made numbers, they begin with a match video!

Today, you are going to build your own sports analytics system. By the end of this mission, your Soccer AI Coach will no longer depend on manually entered statistics—it will create its own statistics directly from match footage. Welcome to the world of **Computer Vision**!

---

## 🎯 Learning Objectives
* ✅ Explain how computers represent videos as sequences of digital images (frames).
* ✅ Read soccer videos frame-by-frame using OpenCV (`cv2`).
* ✅ Upload video files inside Streamlit using `st.file_uploader()`.
* ✅ Detect soccer players inside video frames using YOLO (`ultralytics`).
* ✅ Understand the key difference between **Object Detection** and **Object Tracking**.
* ✅ Convert camera pixel coordinates into real soccer pitch coordinates ($120 \times 80$ yards).
* ✅ Generate professional tactical heatmaps using `mplsoccer`.
* ✅ Ask Gemini AI to analyze player movement as an elite UEFA Pro Analyst.
* ✅ Combine Computer Vision, Data Science, Visualization, and AI into one complete application.

---

## 🟢 Phase 1: Understanding Soccer Heatmaps

Before we teach the computer how to collect movement from video, we first need to learn how to visualize movement on a soccer pitch.

In modern sports analytics, a **Heatmap** shows where a player spends most of their time during a match. Every time a player touches the ball or moves into space, we record their position as an $(X, Y)$ coordinate:
- **X Coordinate (0 to 120 yards):** The length of the pitch (0 = own goal line, 120 = opponent's goal line).
- **Y Coordinate (0 to 80 yards):** The width of the pitch (0 = left touchline, 80 = right touchline).

Let's write a program using `mplsoccer` and `matplotlib` to plot pitch movement:

### 🧰 New Libraries

Today we introduce:

| Library     | Purpose                           |
| ----------- | --------------------------------- |
| mplsoccer   | Professional soccer visualization |
| OpenCV      | Video processing                  |
| YOLO        | Player detection                  |
| Ultralytics | Computer vision models            |
| Gemini API  | Tactical intelligence             |


### Step 1. Your First Soccer Analytics Pitch

Before analyzing videos, we need to understand: How does a computer represent a player's movement?

A human sees:

"Artin moved from defense to attack."

A computer sees coordinates:

**bold text**(x,y)

(20,60)

(30,50)

(50,35)

(90,20)

**Exercise 1.1:** Draw Artin's Movement

Run:

In [21]:
!pip install mplsoccer opencv-python

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch

pitch = Pitch(
    pitch_type="statsbomb"
)

fig, ax = pitch.draw(figsize=(10,7))

x = [10,20,30,45,60,75,90]
y = [60,55,50,40,30,25,20]

pitch.scatter(
    x,
    y,
    ax=ax,
    s=80,
    color="black"
)

plt.title(
"Artin Movement Map"
)

plt.show()

### Step2. Creating Real Player Movement Data

In real applications, coordinates are loaded dynamically from files. We manage movement tracking logs using **Pandas DataFrames** and store them into `.csv` files.

Example:

| frame | x  | y  |
| ----- | -- | -- |
| 1     | 10 | 60 |
| 2     | 20 | 55 |
| 3     | 30 | 50 |


**Exercise 1.2:** Create Movement DataFrame

Run:

In [23]:
import pandas as pd

movement = {

"frame":[1,2,3,4,5],

"x":[10,20,35,50,70],

"y":[60,55,45,35,25]

}

df = pd.DataFrame(movement)

df

### Step3.  Generate a Player Heatmap

Now we answer:

Where does Artin spend most of his time?

In [ ]:
from mplsoccer import Pitch
import matplotlib.pyplot as plt


pitch = Pitch(
pitch_type="statsbomb"
)

fig,ax=pitch.draw()

pitch.kdeplot(
df["x"],
df["y"],
ax=ax,
fill=True
)

pitch.scatter(
df["x"],
df["y"],
ax=ax
)

plt.title(
"Artin Movement Heatmap"
)

plt.show()

**Exercise 1.3:**

Change the coordinates.

Create: Attacking winger

Let's choose a better layout and color for our pitch plot. 

In [ ]:
import matplotlib.pyplot as plt
import mplsoccer
from mplsoccer import Pitch
import numpy as np

# 1. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# 2. Simulate positional data
player = "Artin"
position = "Left Winger"

x_coordinates = [10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105]
y_coordinates = [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]

# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw individual touch points on top
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(f"{player}'s Soccer Heatmap - {position}", fontsize=18, fontweight='bold', pad=15)
plt.show()

### Step 4.  Loading Dataframes from a CSV File

In real applications, coordinates are loaded dynamically from files. We manage movement tracking logs using **Pandas DataFrames** and store them into `.csv` files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Create synthetic player tracking dataset
data = {
    'frame': list(range(1, 11)),
    'x': [18, 20, 21, 25, 30, 42, 55, 68, 72, 85],
    'y': [52, 53, 54, 50, 48, 45, 40, 38, 35, 30]
}

# Save to CSV
df = pd.DataFrame(data)
df.to_csv("player_movement.csv", index=False)
print("✅ CSV file 'player_movement.csv' created successfully!")

# Load CSV and render Kernel Density Estimate (KDE) smooth heatmap
movement_df = pd.read_csv("player_movement.csv")
print("Loaded Data:")
print(movement_df.head())

**Exercise 1.4:** Add 5 additional frames to the synthetic dataset showing the player moving back toward own goal line (lowering $X$ values) and re-generate the CSV.

In [ ]:
## Your code here...

Let's put everything together; loading dataframe(player coordinates) from a `.csv` file and plot it in a pitch. 

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

player = "Artin"
position = "Right Winger"

# 1. Load CSV 
movement_df = pd.read_csv("player_movement.csv")
print("Loaded Data:")
print(movement_df.head())

# 2. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))


# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw individual touch points on top
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(f"{player}'s Soccer Heatmap - {position}", fontsize=18, fontweight='bold', pad=15)
plt.savefig("phase1_heatmap.png", bbox_inches='tight')
plt.show()

**Exercise 1.5:** Modify the colormap in `pitch.heatmap()` from `'Reds'` to `'plasma'` or `'Blues'` and note how it changes the visual contrast.

---

## Phase 2. Enhance Coach_Toolbox Library with Pitch Plot

### Step 1. Developing Pitch Plot function

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch


def pitch_plot (player, position, csv_file):

    # 1. Load CSV    
    movement_df = pd.read_csv(csv_file)
    x_coordinates = movement_df["x"]
    y_coordinates = movement_df["y"]


    # 2. Create a professional soccer pitch layout
    pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
    fig, ax = pitch.draw(figsize=(10, 7))


    # 3. Plot the touches as a heatmap (2D Histogram)
    bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
    pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

    # 4. Draw individual touch points on top
    pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

    plt.title(f"{player}'s Soccer Heatmap - {position}", fontsize=18, fontweight='bold', pad=15)
    plt.savefig("phase1_heatmap.png", bbox_inches='tight')
    plt.show()

# test your function
player = "Artin"
position = "Right Winger"
csv_file = "player_movement.csv"

pitch_plot(player, position, csv_file)

### Step 2. Adding pitch_plot() Function to Coach_Toolbox_v3 Library
Add pitch_plot() function to Coach_Toolbox_v3 library and test it. 

In [17]:

import matplotlib.pyplot as plt
import mplsoccer
from mplsoccer import Pitch
import Coach_Toolbox_v3 as tb

player = "Artin"
position = "Right Winger"
csv_file = "player_movement.csv"

tb.pitch_plot(player, position, csv_file)


### Step 3. Adding Multiple Position Coordinates

In [29]:
import pandas as pd
import Coach_Toolbox_v3 as tb


def create_player_csv(position, filename):

    if position == "Left Winger":
        data = {
            'frame': list(range(1, 21)),
            'x_coordinates' : [10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105],
            'y_coordinates' : [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]
        }

    elif position == "Striker":
         data = {
            'frame': list(range(1, 21)),
            'x_coordinates' : [70,75,80,85,90,95,100,102,105,108,110,112,115,108,104,100,95,90,88,110],
            'y_coordinates' : [35,38,40,42,40,38,35,37,40,42,39,36,40,45,48,50,45,42,38,35]
        }     
    elif position == "Midfielder":
         data = {
            'frame': list(range(1, 21)),
            'x_coordinates' : [35,40,45,50,55,60,65,70,60,55,50,45,40,55,65,75,70,60,50,45],
            'y_coordinates' : [25,30,35,40,45,40,35,30,25,20,25,30,35,45,50,45,40,35,30,25]
        }     
    elif position == "Right Defender":
         data = {
            'frame': list(range(1, 21)),
            'x_coordinates' : [15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110],
            'y_coordinates' : [70,72,68,70,72,74,76,74,72,70,68,70,72,74,76,72,68,65,60,55]
         }  

    # Save to CSV
    df = pd.DataFrame(data)
    df.to_csv("player_movement.csv", index=False)

        
tb.create_player_csv( position="Midfielder", filename="player_movement.csv" )

### Step 4. Building a Soccer Heatmap with `Coach_Toolbox_v3`

Instead of writing dozens of lines of code every time, we can use the functions we created in **Coach_Toolbox_v3**.

In this example, we will complete the task in **three simple steps**:

####  1. Import the toolbox

```python
import Coach_Toolbox_v3 as tb
```

This imports our custom soccer toolbox and gives it the short name **`tb`**, making the functions easier to call.

####  2. Create a sample movement dataset

```python
tb.create_player_csv(
    position="Midfielder",
    filename="player_movement.csv"
)
```

This function automatically generates realistic movement coordinates for a player based on the selected position and saves them into a CSV file named **`player_movement.csv`**.

Behind the scenes, the CSV contains three columns:

- `frame`
- `x_coordinates`
- `y_coordinates`

This simulates the player's movement during a match.

#### 3. Draw the player's heatmap

```python
tb.pitch_plot(
    player="Adam",
    position="Midfielder",
    csv_file="player_movement.csv"
)
```

The `pitch_plot()` function reads the CSV file, places the player's movements onto a professional soccer pitch, and generates a heatmap showing the areas where the player spent the most time during the match.


In [ ]:
import Coach_Toolbox_v3 as tb

tb.create_player_csv(
    position="Midfielder",
    filename="player_movement.csv"
)

tb.pitch_plot(
    player="Adam",
    position="Midfielder",
    csv_file="player_movement.csv"
)

### ✏️ Phase 2 Practice Exercises

**Exercise 2.1:** Change the `position` variable to `"Left Winger"` and re-run the code above. Observe how the heatmap concentration shifts to the top sideline.

---

## 🎥 Phase 3: Reading Soccer Videos with OpenCV

A video is simply a fast sequence of still images called **frames**. OpenCV (`cv2`) allows us to:
1. Open a video file using `cv2.VideoCapture()`.
2. Extract technical parameters: **Resolution** (Width $\times$ Height), **FPS** (Frames Per Second), and **Total Frame Count**.
3. Read individual frames as NumPy arrays for image processing.

### Step 1. Understanding Video with OpenCV

A video is simply:

`Frame 1
 Frame 2
 Frame 3
 ...
 Frame 100`

OpenCV allows us to read these frames.

In [17]:
# Uploading video in google colab for analysis

from google.colab import files

uploaded = files.upload()

Saving Video1.mp4 to Video1.mp4


In [19]:
# Open the video file using `cv2.VideoCapture()`

import cv2

cap = cv2.VideoCapture("Video1.mp4")

#Extract technical parameters: **Resolution** (Width $\times$ Height), **FPS** (Frames Per Second), and **Total Frame Count**.
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = frame_count / fps if fps > 0 else 0

print("📹 Video Information")
print("--------------------")
print(f"Resolution : {width} x {height} pixels")
print(f"FPS        : {fps:.2f} frames/sec")
print(f"Frames     : {frame_count}")
print(f"Duration   : {duration:.2f} seconds")

Next, we save first frame as a jpg picture. 

In [ ]:
# Save first frame to disk
success, frame = cap.read()
if success:
    cv2.imwrite("first_frame.jpg", frame)
    print("✅ Saved first frame as 'first_frame.jpg'")

#### 1. Read the first frame

```python
success, frame = cap.read()
```

The `cap.read()` function reads **one frame** from the video.

It returns two values:

- **`success`** — A Boolean value (`True` or `False`) indicating whether a frame was successfully read.
- **`frame`** — The actual image stored as a NumPy array.

For example:

```python
success = True
frame = [
    [
        [255, 120, 80],
        [0, 200, 100],
        [50, 50, 255]
    ],
    [
        [100, 150, 200],
        [20, 80, 120],
        [255, 255, 0]
    ]
]
```
Each small group of three numbers represents **one pixel**.

If the video cannot be read (for example, it is empty or corrupted), then:

```python
success = False
```


Lets develop a function that open a video file and extract the technical parameters!

In [ ]:
import cv2

def inspect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video file: {video_path}")
        return
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    print("📹 Video Metadata Summary")
    print("------------------------")
    print(f"Resolution : {width} x {height} pixels")
    print(f"FPS        : {fps:.2f} frames/sec")
    print(f"Frames     : {frame_count}")
    print(f"Duration   : {duration:.2f} seconds")
    
    # Save first frame to disk
    success, frame = cap.read()
    if success:
        cv2.imwrite("first_frame.jpg", frame)
        print("✅ Saved first frame as 'first_frame.jpg'")
    
    cap.release()


inspect_video("Video1.mp4")



---

## 💻 Phase 4: Building a Soccer Video Upload Interface with Streamlit



### Step 1. Streamlit Soccer Vision App

Now we convert our notebook into a professional application.

First install the required libraries:

`pip install mplsoccer opencv-python`

Create a Streamlit app named: `Soccer_Vision_App.py`

In [ ]:
import streamlit as st


st.title(
"⚽ Artin FC Soccer Vision Lab"
)


st.write(
"""
AI-powered soccer movement analysis system
"""
)

#Now add select the player and position:

player_name = st.text_input("Player Name")

position = st.selectbox(
"Position",
[
"Forward",
"Midfielder",
"Defender"
]
)

### Step 2. Upload and Open Soccer Video
Now your app can receive match footage.
To upload and read videos dynamically in a web app, we combine Streamlit's `st.file_uploader()` with Python's built-in `tempfile` module. OpenCV needs a physical file path to read video frames.

In [ ]:
import streamlit as st
import cv2
import tempfile


video = st.file_uploader(
"Upload Soccer Video",
type=["mp4"]
)

# if uploaded

if video:

    st.video(video)

    # Save uploaded video
    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )

    temp_file.write(video.read())
    temp_file.close()
 
  # Open video
    cap = cv2.VideoCapture(temp_file.name)
    


### Step 3. Extracting Video Information and Reading the First Frame
 Once OpenCV reads video frames, we can extract the video information and read the first frame as follows:

In [ ]:
import streamlit as st
import cv2
import tempfile


video = st.file_uploader("Upload Video", type=["mp4"])

if video:

    st.video(video)

    # Save uploaded video
    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )

    temp_file.write(video.read())
    temp_file.close()

    # Open video
    cap = cv2.VideoCapture(temp_file.name)

    # Video information
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read first frame
    ret, frame = cap.read()
    
    # Display video information in 4 columns

    st.subheader("📹 Video Information")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Frames", total_frames)
    col2.metric("FPS", round(fps, 1))
    col3.metric("Width", width)
    col4.metric("Height", height)

    cap.release()

## 🤖 Phase 5: YOLO — Object Detection & Object Tracking

### Object Detection vs. Object Tracking
- **Object Detection:** Identifies *what* is in an image frame (e.g., "Person at $[x1, y1, x2, y2]$"). In every frame, it treats detections independently.
- **Object Tracking:** Assigns a persistent **ID number** (Track ID) to each specific individual so they can be followed across consecutive video frames over time.

We use the **YOLOv8** model from `ultralytics` to perform real-time tracking.

`YOLO` (You Only Look Once) can detect:

- players
- balls
- referees
- objects

Install `pip install ultralytics`

In [31]:
# Load model:
from ultralytics import YOLO

model=YOLO(
"yolov8n.pt"
)

Here is the complete app with YOLO detection.

In [ ]:
import streamlit as st
import cv2
import tempfile
from ultralytics import YOLO


# Load YOLO model
model = YOLO("yolov8n.pt")

video = st.file_uploader("Upload Video", type=["mp4"])

if video:

    st.video(video)

    # Save uploaded video
    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )

    temp_file.write(video.read())
    temp_file.close()

    # Open video
    cap = cv2.VideoCapture(temp_file.name)

    # Video information
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read first frame
    ret, frame = cap.read()
    
    # Display video information in 4 columns

    st.subheader("📹 Video Information")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Frames", total_frames)
    col2.metric("FPS", round(fps, 1))
    col3.metric("Width", width)
    col4.metric("Height", height)

    if ret:

        st.write("Running YOLO on first uploaded video frame")

        # YOLO detection
        results = model(frame)

        result = results[0]

        # Draw detections
        annotated = result.plot()

        st.image(
            annotated,
            channels="BGR"
        )

        # Display detected objects
        for box in result.boxes:

            cls = int(box.cls[0])
            conf = float(box.conf[0])

            st.write(
                f"{result.names[cls]}: {conf:.2f}"
            )

    cap.release()

Currently YOLO can answer:

"Where are the people in this frame?"

But it cannot answer:

"Which player is Artin?"

For that we need:

1. Object tracking
   - Keep the same player ID across frames

2. Jersey number recognition
   - Detect player number 4

3. Player identity mapping
   - Player ID → Artin

The next mission will extend YOLO from detection to soccer intelligence.


## 🟢 Phase 6: Coordinate Mapping — From Pixel Space to Pitch Space

Camera coordinates are measured in **Pixels** ($1280 \times 720$). Tactical maps require dimensions measured in **Yards** ($120 \times 80$).

A camera sees:

1280 x 720 pixels

Soccer uses:

120 x 80 yards

We need conversion.

In [34]:
def convert_coordinates(pixel_x, pixel_y):

    soccer_x = (pixel_x/1280)*120

    soccer_y = (pixel_y/720)*80

    return soccer_x,soccer_y

The complete code

In [ ]:
import streamlit as st
import cv2
import tempfile
from ultralytics import YOLO

def convert_coordinates(pixel_x, pixel_y, frame_width, frame_height):

    soccer_x = (pixel_x / frame_width) * 120
    soccer_y = (pixel_y / frame_height) * 80

    return soccer_x, soccer_y


st.title(
"⚽ Artin FC Soccer Vision Lab"
)

st.write(
"""
AI-powered soccer movement analysis system
"""
)

player_name = st.text_input("Player Name")

position = st.selectbox(
"Position",
[
"Forward",
"Midfielder",
"Defender"
]
)

import streamlit as st
import cv2
import tempfile
from ultralytics import YOLO

# Load YOLO model
model = YOLO("yolov8n.pt")

video = st.file_uploader("Upload Video", type=["mp4"])

if video:

    st.video(video)

    # Save uploaded video
    temp_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )

    temp_file.write(video.read())
    temp_file.close()

    # Open video
    cap = cv2.VideoCapture(temp_file.name)

    # Video information
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read first frame
    ret, frame = cap.read()

    st.subheader("📹 Video Information")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Frames", total_frames)
    col2.metric("FPS", round(fps, 1))
    col3.metric("Width", width)
    col4.metric("Height", height)

    if ret:

        st.write("Running YOLO on first uploaded video frame")

        # YOLO detection
        results = model(frame)

        result = results[0]

        # Draw detections
        annotated = result.plot()

        st.image(
            annotated,
            channels="BGR"
        )

        #Add coordinate extraction inside your YOLO loop
        for box in result.boxes:

            cls = int(box.cls[0])
            conf = float(box.conf[0])

            object_name = result.names[cls]

            st.write(f"{object_name}: {conf:.2f}")


          # Bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0]


          # Convert tensor values to numbers
            x1 = int(x1)
            y1 = int(y1)
            x2 = int(x2)
            y2 = int(y2)


    # Calculate object center
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2


    # Convert to soccer coordinates
            soccer_x, soccer_y = convert_coordinates(
            center_x,
            center_y,
            width,
            height)


            st.write(f"""{object_name} position:

            Pixel:
            ({center_x:.0f}, {center_y:.0f})

            Soccer field:
            ({soccer_x:.1f}, {soccer_y:.1f}) yards
            """
        )

            cap.release()



# Final Boss Challenge 🏆  


In this challenge, you will build the first version of the **Artin FC AI Tactical Center**.

Your goal is to create a computer vision system that can look at a soccer video frame and understand **where players are located on the field**.

Your application must include:

---

## 1. Upload a Soccer Video 🎥

The user should be able to:

- Upload a real soccer match clip
- Extract frames from the video
- Display the selected frame

Example:

```
Upload → Video → First Frame
```



In [ ]:
# Paste Your code here...

## Step 2. Detect Players Using YOLO 🤖

Your AI system should:

- Detect all players in the frame
- Draw bounding boxes around detected players
- Display confidence scores

Example output:

```
Player 1   confidence: 0.91
Player 2   confidence: 0.87
Player 3   confidence: 0.84
```

In [ ]:
# Paste Your code here...


## Step 3. Convert Pixel Location to Soccer Coordinates ⚽

For each detected player:

Convert:

```
Image Coordinates
(x_pixel, y_pixel)
```

into:

```
Soccer Field Coordinates
(x_pitch, y_pitch)
```

Example:

```
Player detected at:

Pixel:
(850, 430)

Converted to:

Pitch:
(53.2, 28.5)
```


In [ ]:
# Paste Your code here...


## Step 4. Create a Player Location Table 📊

Your system should create a dataset like:

| Player ID | Pixel X | Pixel Y | Pitch X | Pitch Y |
|---|---|---|---|---|
| 1 | 850 | 430 | 53.2 | 28.5 |
| 2 | 600 | 520 | 37.5 | 40.2 |

Save this information as:

```
player_locations.csv
```

In [ ]:
# Paste Your code here...

Here is a sample code to visualize the player positions collected in Step 4. You can modify the number of objects based on the number of players detected in your sample video. You have to find a sample video and test your entire code with it.

In [ ]:
import pandas as pd


# Create empty lists to store information
player_ids = []
pixel_x = []
pixel_y = []
pitch_x = []
pitch_y = []


# Add detected players
# Example: YOLO detected 3 players

player_ids.append(1)
pixel_x.append(850)
pixel_y.append(430)
pitch_x.append(53.2)
pitch_y.append(28.5)


player_ids.append(2)
pixel_x.append(600)
pixel_y.append(520)
pitch_x.append(37.5)
pitch_y.append(40.2)


player_ids.append(3)
pixel_x.append(920)
pixel_y.append(380)
pitch_x.append(58.1)
pitch_y.append(25.4)



# Create a table
player_locations = pd.DataFrame({

    "Player ID": player_ids,
    "Pixel X": pixel_x,
    "Pixel Y": pixel_y,
    "Pitch X": pitch_x,
    "Pitch Y": pitch_y

})

player_locations.to_csv(
    "player_locations.csv",
    index=False
)

st.subheader("📊 Player Location Table")

st.dataframe(
    player_locations,
    use_container_width=True
)


---

## 🏆 Mission 12 Complementary Preview

Congratulations! 🏆 You have completed the first part of Mission 12 and built an **AI Soccer Analytics System** that extracts valuable player data from raw video files for further analysis.

In this mission, you learned how an AI system can:
- Read soccer videos,
- Detect players using computer vision,
- Convert player locations into field coordinates, and
- Create structured datasets for analysis.

In the next mission, you will take the next step by learning how to transform video detection into **player tracking** and use the collected movement data to support **intelligent coaching decisions**. ⚽🤖